## 🎯✍️ OverWriting Strategies (Estrategias de Sobre-Escritura)

Hasta este momento hemos aprendido diferentes formas de escribir información en **Delta Tables**, tanto a partir de **fuentes externas** como **fuentes internas** dentro de Databricks. Sin embargo, surge una pregunta importante:

> 🤔 ¿Siempre debemos escribir los datos de la misma manera?

La respuesta es **no**.

En los capítulos anteriores conocimos distintos mecanismos para persistir información en Delta Lake, pero estos representaban únicamente el punto de partida.

En escenarios reales de Data Engineering, la estrategia de escritura dependerá de las necesidades del proceso, el volumen de datos, la frecuencia de actualización y las buenas prácticas de optimización que deseemos implementar.

Por ello, Databricks ofrece diferentes estrategias de sobre-escritura, cada una diseñada para resolver un escenario específico.

---

### 🚀 Estrategias que veremos

Durante este bloque revisaremos las siguientes estrategias:

* 🗑️ **Drop & Recreate Table**
* 🔄 **INSERT OVERWRITE**
* 🔀 **MERGE + 📈 Slowly Changing Dimensions (SCD)** 

Cada una presenta ventajas, limitaciones y casos de uso particulares, por lo que elegir la estrategia adecuada será un aspecto fundamental dentro de cualquier pipeline de Data Engineering.

---

### 🎯 En este capítulo...

Comenzaremos estudiando la estrategia más sencilla: **Drop & Recreate Table**, comprendiendo cómo funciona, cuándo resulta conveniente utilizarla y cuáles son sus principales ventajas y limitaciones.


### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("14WritingStrategies1").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")


## 🗑️ DROP & RECREATE TABLE

Pues bien, comenzaremos con la primera estrategia de sobreescritura: **Drop & Recreate Table**.

Como su propio nombre indica, esta estrategia consiste en:

1. 🗑️ Eliminar la tabla que tenemos actualmente.
2. 🔄 Crear nuevamente la tabla.
3. 📥 Escribir en ella la nueva información que queremos almacenar.

En otras palabras, en lugar de modificar el contenido de la Delta Table existente, **eliminamos su definición actual y construimos una nueva tabla desde cero**.

---

#### ⚠️ ¿Cuál es el problema?

Aunque esta estrategia puede parecer sencilla, **no es la más recomendable cuando queremos aprovechar las capacidades de Delta Lake**.

Al eliminar la tabla original, también perdemos el historial asociado a ella.

Esto significa que dejamos de aprovechar características como:

* 🕐 **Time Travel**
* 📜 **Historial de transacciones**
* 🔄 **Versionamiento de la tabla**

Por lo tanto, aunque `DROP & RECREATE` puede permitirnos obtener rápidamente una tabla con información actualizada, **rompe la continuidad del historial de la Delta Table**.

> 💡 En una arquitectura basada en Delta Lake, debemos tener cuidado con las estrategias que implican eliminar y reconstruir completamente una tabla, debido que podemos perder información histórica que posteriormente podría ser necesaria para auditoría, comparación o recuperación.


In [0]:
### CONFIGURACIÓN DE TABLA INICIAL

"""
    EN ESTE CASO, CUALQUIER FORMA DE CREAR UNA TABLA INICIALMENTE ES VÁLIDA.
    PARA EL EJEMPLO, UTILIZARÉ LA DEFINICIÓN DE LA TABLA ORIGINAL MEDIANTE CTAs.
"""

### PATH DEL ORIGEN DE DATOS
path_interno_source_data_csv = "/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/csv/"

### PASO A). PREPARAR QUERY SQL ( UTILIZAREMOS FORMA DE LECTURA READ_FILES )

display(spark.sql(f"""
                  
-- INICIALMENTE LEEMOS LA INFORMACIÓN DEL CLIENTE 1.

                  SELECT *
                  FROM read_files(
                      '{path_interno_source_data_csv}/sales_client_1.csv',
                      format => 'csv',
                      inferSchema => true,
                      header => true,
                      delimiter => ','
                  )

                  """))

### PASO B). ADJUNTAMOS QUERY A UN CTAs

spark.sql(f"""
-- CREAMOS LA TABLA CON LA INFORMACIÓN DEL CLIENTE 1.          
        CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.ctas_sales_csv_drop_recreate
        AS
        SELECT *
        FROM read_files(
            '{path_interno_source_data_csv}/sales_client_1.csv',
            format => 'csv',
            inferSchema => true,
            header => true,
            delimiter => ','
        )
         
          """)

print("Tabla creada correctamente a partir de un CTAs")

## PASO C). VERIFICAMOS CTAs
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.ctas_sales_csv_drop_recreate"))

### PASO D). VERIFICAR HISTORIAL INICIAL DE LA TABLA
display(spark.sql("DESCRIBE HISTORY catalog_databricks_2026_de.schema_databricks_2026_de.ctas_sales_csv_drop_recreate"))
#### Resultado: Posee solo 1 versión, es decir, solo cuando definimos la tabla a partir de una query (CTAs).

In [0]:
"""
    Ahora bien: ¿Que sucede si necesitamos almacenar la información
    del cliente 2,3,N?. Pues, debemos eliminar la tabla original y
    volver a definirla, pero, apuntando a los datos del cliente 2 +
    del cliente 1.
"""

### PASO 1). ELIMINAR TABLA INICIAL
spark.sql("DROP TABLE catalog_databricks_2026_de.schema_databricks_2026_de.ctas_sales_csv_drop_recreate")
print("Tabla eliminada correctamente")

### PASO 2). VOLVER A CREAR LA TABLA MEDIANTE EL CTAs

spark.sql(f"""
-- CREAMOS LA TABLA CON LA INFORMACIÓN DEL CLIENTE 1 y 2, GRACIAS AL WILCARD *.          
        CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.ctas_sales_csv_drop_recreate
        AS
        SELECT *
        FROM read_files(
            '{path_interno_source_data_csv}/sales_client_*.csv',
            format => 'csv',
            inferSchema => true,
            header => true,
            delimiter => ','
        )
         
          """)
print("Tabla creada correctamente a partir de un CTAs")

## PASO C). VERIFICAMOS CTAs
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.ctas_sales_csv_drop_recreate"))

### PASO D). VERIFICAR HISTORIAL INICIAL DE LA TABLA
display(spark.sql("DESCRIBE HISTORY catalog_databricks_2026_de.schema_databricks_2026_de.ctas_sales_csv_drop_recreate"))
#### Resultado: Posee solo 1 versión, es decir, la tabla al eliminarse perdió su versión inicial y solo crea una versión al volver a re-crearla. No es recomendable.